# Exam: Reverse-Engineering Implicit Chain-of-Thought for Multi-Digit Multiplication

This exam tests your understanding of the research on why Transformers fail at multi-digit multiplication and how Implicit Chain-of-Thought (ICoT) training enables successful learning.

---

## Key Knowledge Points

### 1. Research Motivation and Goals
- **Primary Question**: Why do Transformers fail at seemingly simple tasks like 4×4 digit multiplication despite having sufficient capacity?
- **Hypothesis**: By reverse-engineering successful ICoT models, we can understand what standard fine-tuning (SFT) models lack
- **Objectives**: Identify long-range dependencies, computational mechanisms, geometric representations, and learning dynamics

### 2. Data Format and Training
- **Task**: 4×4 digit multiplication (smallest setting where SFT fails but ICoT succeeds)
- **Digit Order**: Least-significant digit first (reversed from standard notation)
- **ICoT Curriculum**: 8 chain-of-thought tokens removed per epoch over 13 epochs
- **Training Data**: 80,800 training samples, 1,000 validation/test samples each

### 3. Model Architecture and Training Procedures
- **Minimal Architecture**: 2 layers, 4 heads (2L4H) with d=768 embedding dimension
- **ICoT Training**: Gradual removal of explicit CoT tokens forces internalization
- **Standard Fine-Tuning (SFT)**: Fails even with 12L8H scaling - < 1% accuracy
- **Auxiliary Loss**: Linear probes predict running sums ĉk, achieves 99% accuracy

### 4. Performance Results
- **ICoT (2L4H)**: 100% accuracy
- **SFT (2L4H)**: < 1% example accuracy, ~81% digit-level accuracy
- **SFT (12L8H)**: Still < 1% (scaling doesn't help)
- **Auxiliary Loss (2L4H)**: 99% accuracy

### 5. Discovered Mechanisms
- **Attention Tree Structure**:
  - Layer 1 (Caching): Stores pairwise products aᵢbⱼ in hidden states
  - Layer 2 (Retrieval): Attends to cached values to compute output digits
- **Minkowski Sums**: Attention outputs form nested geometric structures
- **Fourier Basis**: Digits represented using k ∈ {0,1,2,5} frequencies plus parity
- **Pentagonal Prism**: 3D PCA reveals two parallel pentagons (even/odd digits)

### 6. Evidence of Long-Range Dependencies
- **Logit Attribution**: ICoT shows correct pattern (aᵢ, bⱼ affect cₖ when k ≥ i+j)
- **Linear Probes**: ICoT encodes running sums ĉk with MAE ~1-2, SFT has MAE ~30-110
- **Intermediate Value ĉk**: Accumulated sum up to position k (not final digit)

### 7. Learning Dynamics
- **SFT Failure Pattern**:
  - Learns c₀, c₁ first (local patterns)
  - Then c₇ (last digit, also local)
  - Middle digits c₃-c₆ plateau (require long-range dependencies)
  - Gradient norms drop to zero after early digits learned
- **Root Cause**: Optimization landscape issue, not capacity limitation

### 8. Geometric Representations
- **Fourier Fit Quality**: 
  - Embeddings E: R² = 0.84 (k=0,1,2,5), 1.0 (all k)
  - Final hidden h^L: R² = 0.99 (k=0,1,2,5), 1.0 (all k)
- **Self-Similarity**: Minkowski sums create fractal-like nested clusters

### 9. Key Insights
- **Three Critical Structures**: Binary attention trees, Minkowski sums, Fourier bases
- **Scaling Alone Insufficient**: Problem is optimization, not capacity
- **Simple Inductive Biases Help**: Auxiliary loss demonstrates minimal supervision suffices
- **ICoT Provides Learning Path**: Guides model away from local optima

### 10. Implications
- Multi-step reasoning tasks need specific computational structures
- Standard training may fail even with sufficient capacity
- Mechanistic interpretability reveals what failed models lack
- Process supervision (explicit or implicit) appears necessary

---

## Exam Questions

This exam contains:
- **Multiple-choice questions**: Test factual understanding
- **Free-generation questions**: Test deeper reasoning and application
- **Code questions**: Verify computational understanding of key claims

Please answer all questions based on the documentation provided.

---

## Code Questions

The following code questions ask you to implement verification of key research claims.

### Code Question 1: Fourier Basis R² Verification (CQ1)

**Task**: Verify the claim about Fourier basis R² fits.

Implement a function that:
1. Generates 10 random digit embedding vectors (simulating embeddings for digits 0-9 with dimension d=768)
2. Constructs the Fourier basis matrix with frequencies k ∈ {0,1,2,5} plus parity
3. Computes the R² fit
4. Demonstrates that structured embeddings (using the Fourier basis) achieve near-perfect R² while random embeddings achieve much lower R²

The Fourier basis for digit n is:
```
Φ(n) = [1, cos(2πn/10), sin(2πn/10), cos(2πn/5), sin(2πn/5), (-1)^n]
```

**Expected Output**: 
- R² for structured embeddings (built from Fourier basis): ≈ 1.0
- R² for random embeddings: < 0.5
- Print both values to demonstrate the difference

In [ ]:
import numpy as np
import torch

# Set seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# TODO: Implement Fourier basis construction
def construct_fourier_basis():
    """
    Construct the Fourier basis matrix for digits 0-9.
    Returns: matrix of shape (10, 6) where each row is Φ(n) for digit n
    Φ(n) = [1, cos(2πn/10), sin(2πn/10), cos(2πn/5), sin(2πn/5), (-1)^n]
    """
    # YOUR CODE HERE
    pass

# TODO: Implement R² computation
def compute_r_squared(embeddings, fourier_basis):
    """
    Compute R² fit of embeddings to Fourier basis.
    
    Args:
        embeddings: (10, d) matrix of digit embeddings
        fourier_basis: (10, 6) Fourier basis matrix
    
    Returns:
        r_squared: median R² across all d dimensions
    
    For each dimension i of embeddings:
        1. Fit: C_i = argmin_C ||embeddings[:, i] - fourier_basis @ C||²
        2. Compute: R²_i = 1 - ||embeddings[:, i] - fourier_basis @ C_i||² / ||embeddings[:, i] - mean||²
    Return: median(R²_i)
    """
    # YOUR CODE HERE
    pass

# TODO: Create structured embeddings from Fourier basis
def create_structured_embeddings(fourier_basis, d=768):
    """
    Create structured embeddings by projecting Fourier basis to d dimensions.
    
    Args:
        fourier_basis: (10, 6) Fourier basis
        d: embedding dimension
    
    Returns:
        embeddings: (10, d) where embeddings = fourier_basis @ random_projection
    """
    # YOUR CODE HERE
    pass

# TODO: Run the verification
# 1. Construct Fourier basis
# 2. Create structured embeddings
# 3. Create random embeddings
# 4. Compute R² for both
# 5. Print results

# YOUR CODE HERE
print("TODO: Implement and run Fourier basis R² verification")

### Code Question 2: Attention Tree Caching/Retrieval Simulation (CQ2)

**Task**: Simulate and verify the attention tree caching/retrieval mechanism.

Implement a simplified version where:
1. Layer 1 'caches' pairwise products aᵢ * bⱼ in a dictionary keyed by timestep
2. Layer 2 'retrieves' these values using attention-like indexing to compute a specific output digit cₖ
3. Demonstrate that this tree structure correctly computes c₃ for the example: 8331 × 5015 (input format: 1338 * 5105)
4. Show which cached products are retrieved

**Mathematical Background**:
- c₃ requires partial products: a₃b₀, a₂b₁, a₁b₂, a₀b₃, plus carry from c₂
- c₂ requires: a₂b₀, a₁b₁, a₀b₂, plus carry from c₁
- Each digit cₖ = (Σ aᵢbⱼ for i+j=k, plus carry from k-1) mod 10

**Expected Output**:
- Print the cached products at each timestep
- Print which products are retrieved for computing c₃
- Compute and print c₃ = 7 (from 8331 × 5015 = 41797665)

In [ ]:
import numpy as np

# Example: 8331 × 5015 (reversed: 1338 * 5105)
a = [1, 3, 3, 8]  # 8331 reversed
b = [5, 1, 0, 5]  # 5015 reversed

# TODO: Implement Layer 1 caching
def layer1_cache_products(a, b):
    """
    Simulate Layer 1: cache pairwise products aᵢ * bⱼ.
    
    Args:
        a: list of 4 digits (operand 1, reversed)
        b: list of 4 digits (operand 2, reversed)
    
    Returns:
        cache: dict mapping timestep -> list of (i, j, product) tuples cached at that step
    
    Example strategy: cache a₀b₀ at t=0, a₁b₀ and a₀b₁ at t=1, etc.
    """
    # YOUR CODE HERE
    pass

# TODO: Implement Layer 2 retrieval
def layer2_retrieve_and_compute(cache, k, a, b):
    """
    Simulate Layer 2: retrieve cached products and compute cₖ.
    
    Args:
        cache: dict from layer1_cache_products
        k: which output digit to compute
        a, b: operand digits
    
    Returns:
        c_k: the k-th output digit
        retrieved: list of (i, j, product) tuples retrieved for this computation
    
    Algorithm:
        1. Retrieve all cached products where i+j ≤ k
        2. Sum products where i+j = k
        3. Add carry from c_{k-1}
        4. Return digit (sum mod 10) and carry (sum // 10)
    """
    # YOUR CODE HERE
    pass

# TODO: Run the simulation
# 1. Cache products in Layer 1
# 2. Retrieve and compute c₃ in Layer 2
# 3. Print cached products
# 4. Print retrieved products for c₃
# 5. Print computed c₃ value

# YOUR CODE HERE
print("TODO: Implement and run attention tree simulation")

### Code Question 3: Logit Attribution Dependency Pattern (CQ3)

**Task**: Verify the logit attribution dependency pattern.

For a sample 4×4 multiplication, implement a function that:
1. Computes which input digit positions (aᵢ, bⱼ) should affect which output digit cₖ according to the mathematical structure of multiplication
2. Creates a dependency matrix showing this structure
3. Validates that cₖ depends on partial products where i+j ≤ k
4. Shows that the strongest dependencies occur when i+j = k (direct contribution without carry)

**Expected Output**:
- Dependency matrix of shape (8 input positions, 8 output positions)
- Input positions: [a₀, a₁, a₂, a₃, b₀, b₁, b₂, b₃]
- Output positions: [c₀, c₁, c₂, c₃, c₄, c₅, c₆, c₇]
- Matrix values: 2 for strong dependency (i+j=k), 1 for weak dependency (i+j<k via carry), 0 for no dependency
- Visualize as heatmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# TODO: Implement dependency computation
def compute_dependency_matrix():
    """
    Compute the mathematical dependency structure for 4×4 multiplication.
    
    Returns:
        dependency_matrix: (8, 8) matrix where:
            - Rows: input positions [a₀, a₁, a₂, a₃, b₀, b₁, b₂, b₃]
            - Cols: output positions [c₀, c₁, c₂, c₃, c₄, c₅, c₆, c₇]
            - Values: 2 for strong (i+j=k), 1 for weak (i+j<k), 0 for none
    
    Logic:
        - For input aᵢ and output cₖ: check all j where (i,j) contributes to cₖ
        - For input bⱼ and output cₖ: check all i where (i,j) contributes to cₖ
        - Direct contribution (i+j=k): weight 2
        - Indirect via carry (i+j<k): weight 1
        - No contribution (i+j>k): weight 0
    """
    # YOUR CODE HERE
    pass

# TODO: Visualize the dependency matrix
def visualize_dependencies(dependency_matrix):
    """
    Create a heatmap of the dependency matrix.
    """
    # YOUR CODE HERE
    pass

# TODO: Validate key properties
def validate_dependency_properties(dependency_matrix):
    """
    Validate that the dependency matrix satisfies expected properties:
    1. c₀ depends only on a₀ and b₀ (should have strong dependency)
    2. c₇ depends on all inputs (should have at least weak dependencies)
    3. Middle digits (c₃, c₄) have dependencies spanning multiple i,j pairs
    4. Strongest dependencies (value=2) occur when i+j=k
    
    Print validation results.
    """
    # YOUR CODE HERE
    pass

# TODO: Run the analysis
# 1. Compute dependency matrix
# 2. Visualize as heatmap
# 3. Validate properties
# 4. Print summary statistics

# YOUR CODE HERE
print("TODO: Implement and run dependency analysis")

---

## End of Exam

For the complete exam questions (including multiple-choice and free-generation questions), please refer to `exam_icot.json`.

For code solutions and auto-grading, see `exam_code_solutions.ipynb` (instructor version only).